# 🏆 Simulation Coupe du Monde 2026
## Système de prédiction basé sur des données réelles

> **Objectif :** Simuler l'intégralité de la CdM 2026 — phase de groupes, Round of 32, quarts, demis, finale — à partir d'un modèle de score composite et d'une simulation Monte Carlo (10 000 itérations).
|

### Format CdM 2026
- 48 équipes · 12 groupes de 4 · Top 2 de chaque groupe + 8 meilleurs 3ᵉ = **32 équipes**
- Round of 32 → 1/4 de finale → 1/2 finale → Finale (MetLife Stadium, 19 juillet 2026)

---

## 1. Importation des bibliothèques

Import des bibliothèques principales utilisées dans le notebook :

- `pandas` pour la manipulation de données
- `numpy` pour les calculs numériques

(Exécuter la cellule pour vérifier les versions si besoin.)

In [85]:
import pandas as pd
import numpy as np
import itertools
import random

## 2. Chargement des données

Lecture des fichiers sources nécessaires à la simulation et affectation aux DataFrame :

- `valeurs_marchandes.csv` → `df_valeurs`
- `resultats_depuis_2021.csv` → `df_resultats`
- `meilleure_defense.csv` → `df_defenses`
- `classement_fifa.csv` → `df_classement`
- `buts_marques_depuis_2021.csv` → `df_buts`

Exécuter la cellule pour charger les données.

In [86]:
df_valeurs = pd.read_csv('valeurs_marchandes.csv')
df_resultats = pd.read_csv('resultats_depuis_2021.csv')
df_defenses = pd.read_csv('meilleure_defense.csv')
df_classement= pd.read_csv('classement_fifa.csv')
df_buts = pd.read_csv('buts_marques_depuis_2021.csv')

## 3. Fusion et préparation des données

Construction du `DataFrame` maître `df_master` en fusionnant les sources sur la colonne `team` :

- `df_classement` × `df_valeurs` → `df_intermediaire`
- + `df_resultats` → `df_int`
- + `df_defenses` → `df_int_2`
- + `df_buts` → `df_master`

Affichage des premières lignes pour vérification.

In [87]:
df_intermediaire=df_classement.merge(df_valeurs,on='team')
df_int=df_intermediaire.merge(df_resultats,on='team')
df_int_2=df_int.merge(df_defenses,on='team')
df_master=df_int_2.merge(df_buts,on='team')
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021
0,France,1,1440000000,55,38,9,8,44,102
1,Espagne,2,1390000000,54,40,8,6,47,107
2,Argentine,3,755000000,57,43,8,6,42,112
3,Angleterre,4,1720000000,57,37,11,9,55,108
4,Portugal,5,920000000,50,33,9,8,53,84


## 4. Calcul des indicateurs de performance

Création de nouvelles colonnes normalisées dans `df_master` :

- `taux_victoires` = `victoires` / `matchs_officiels_joues`
- `taux_invincibilite` = (`victoires` + `nuls`) / `matchs_officiels_joues`
- `buts_encaisses_par_match` = `buts_encaisses_depuis_2021` / `matchs_officiels_joues`
- `buts_marques_par_match` = `buts_marques_depuis_2021` / `matchs_officiels_joues`

Afficher `df_master.head()` pour valider.

In [88]:
df_master["taux_victoires"]=df_master["victoires"]/df_master["matchs_officiels_joues"]
df_master["taux_invincibilite"]=(df_master["victoires"]+df_master['nuls'])/df_master["matchs_officiels_joues"]
df_master["buts_encaisses_par_match"]=df_master["buts_encaisses_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master["buts_marques_par_match"]=df_master["buts_marques_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000


## 5. Normalisation du classement FIFA

Création de `score_fifa_norme` : normalisation inversée de `classement_fifa` (un classement plus bas → meilleur score normalisé). On affiche ensuite `df_master.head()` pour contrôle.

In [89]:
df_master['score_fifa_norme']=(max(df_master['classement_fifa'])-df_master['classement_fifa'])/(max(df_master['classement_fifa'])-min(df_master['classement_fifa']))
df_master.head()   

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match,score_fifa_norme
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545,1.000000
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481,0.988095
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912,0.976190
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737,0.964286
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000,0.952381


## 6. Transformation et normalisation de la valeur marchande

Calcul de `valeur_log` = log10(`valeur_marchande_euros`) puis normalisation en `valeur_log_norme`. Cette transformation réduit l'étalement des valeurs extrêmes.

In [90]:
df_master['valeur_log']=np.log10(df_master['valeur_marchande_euros'])
df_master['valeur_log_norme']=(df_master['valeur_log']-min(df_master['valeur_log']))/(max(df_master['valeur_log'])-min(df_master['valeur_log']))
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match,score_fifa_norme,valeur_log,valeur_log_norme
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545,1.000000,9.158362,0.968148
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481,0.988095,9.143015,0.961812
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912,0.976190,8.877947,0.852399
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737,0.964286,9.235528,1.000000
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000,0.952381,8.963788,0.887832


## 7. Solidité défensive normalisée

Calcul de `solidite_defensive` : on inverse `buts_encaisses_par_match` puis on normalise pour obtenir un score où une valeur plus élevée signifie meilleure solidité.

In [91]:
df_master['solidite_defensive']=(max(df_master['buts_encaisses_par_match'])-df_master['buts_encaisses_par_match'])/(max(df_master['buts_encaisses_par_match'])-min(df_master['buts_encaisses_par_match']))
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match,score_fifa_norme,valeur_log,valeur_log_norme,solidite_defensive
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545,1.000000,9.158362,0.968148,0.992537
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481,0.988095,9.143015,0.961812,0.984222
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912,0.976190,8.877947,0.852399,1.000000
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737,0.964286,9.235528,1.000000,0.973051
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000,0.952381,8.963788,0.887832,0.961816


## 8. Normalisation des buts marqués

Normalisation de `buts_marques_par_match` en `buts_marques_par_match_norme` pour rendre les valeurs comparables entre équipes.

In [92]:
df_master['buts_marques_par_match_norme']=(df_master['buts_marques_par_match']-min(df_master['buts_marques_par_match']))/(max(df_master['buts_marques_par_match'])-min(df_master['buts_marques_par_match']))
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match,score_fifa_norme,valeur_log,valeur_log_norme,solidite_defensive,buts_marques_par_match_norme
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545,1.000000,9.158362,0.968148,0.992537,0.649658
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481,0.988095,9.143015,0.961812,0.984222,0.723776
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912,0.976190,8.877947,0.852399,1.000000,0.714101
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737,0.964286,9.235528,1.000000,0.973051,0.673125
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000,0.952381,8.963788,0.887832,0.961816,0.547740


## 9. Indices d'attaque et de défense

Calcul des indices synthétiques :

- `puissance_attaque` : combinaison pondérée des buts normés, taux de victoires, valeur marchande normalisée et score FIFA.
- `puissance_defense` : combinaison pondérée de la solidité défensive, taux d'invincibilité et score FIFA.

Afficher `df_master.head()` pour vérification.

In [93]:
df_master['puissance_attaque']=df_master['buts_marques_par_match_norme']*0.4+df_master['taux_victoires']*0.3+df_master['valeur_log_norme']*0.2+df_master['score_fifa_norme']*0.1
df_master['puissance_defense']=df_master['solidite_defensive']*0.5+df_master['taux_invincibilite']*0.3+df_master['score_fifa_norme']*0.2
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match,score_fifa_norme,valeur_log,valeur_log_norme,solidite_defensive,buts_marques_par_match_norme,puissance_attaque,puissance_defense
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545,1.000000,9.158362,0.968148,0.992537,0.649658,0.760765,0.952632
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481,0.988095,9.143015,0.961812,0.984222,0.723776,0.802905,0.956397
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912,0.976190,8.877947,0.852399,1.000000,0.714101,0.780055,0.963659
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737,0.964286,9.235528,1.000000,0.973051,0.673125,0.760416,0.932014
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000,0.952381,8.963788,0.887832,0.961816,0.547740,0.689900,0.923384


In [94]:
def simuler_match(equipeA,equipeB):
    puissance_attaque_A=df_master[df_master['team']==equipeA]['puissance_attaque'].values[0]
    puissance_attaque_B=df_master[df_master['team']==equipeB]['puissance_attaque'].values[0]
    puissance_defense_A=df_master[df_master['team']==equipeA]['puissance_defense'].values[0]
    puissance_defense_B=df_master[df_master['team']==equipeB]['puissance_defense'].values[0]
    lambda_A=5.0*puissance_attaque_A*(1-puissance_defense_B)
    lambda_B=5.0*puissance_attaque_B*(1-puissance_defense_A)
    score_A=np.random.poisson(lambda_A)
    score_B=np.random.poisson(lambda_B)
    return {equipeA: score_A, equipeB: score_B}

In [95]:
GROUPES = {
    'A': ['Mexique',      'Afrique du Sud',     'Corée du Sud',        'Tchéquie'],
    'B': ['Canada',       'Suisse',             'Qatar',               'Bosnie-Herzégovine'],
    'C': ['Brésil',       'Maroc',              'Haïti',               'Écosse'],
    'D': ['États-Unis',   'Paraguay',           'Australie',           'Turquie'],
    'E': ['Allemagne',    'Curaçao',            "Côte d'Ivoire",       'Équateur'],
    'F': ['Pays-Bas',     'Japon',              'Suède',               'Tunisie'],
    'G': ['Belgique',     'Égypte',             'Iran',                'Nouvelle-Zélande'],
    'H': ['Espagne',      'Cap-Vert',           'Arabie Saoudite',     'Uruguay'],
    'I': ['France',       'Sénégal',            'Norvège',             'Irak'],
    'J': ['Argentine',    'Algérie',            'Autriche',            'Jordanie'],
    'K': ['Portugal',     'Congo DR',           'Ouzbékistan',         'Colombie'],
    'L': ['Angleterre',   'Croatie',            'Ghana',               'Panama'],
}
stats_groupes = {groupe: {equipe: {'points': 0, 'buts_pour': 0, 'buts_encaisses': 0,'diff_buts': 0} for equipe in equipes} for groupe, equipes in GROUPES.items()}

In [96]:
for groupe, equipes in GROUPES.items():
    for equipeA, equipeB in itertools.combinations(equipes, 2):
        resultat = simuler_match(equipeA, equipeB)
        score_A = resultat[equipeA]
        score_B = resultat[equipeB]
        
        stats_groupes[groupe][equipeA]['buts_pour'] += score_A
        stats_groupes[groupe][equipeA]['buts_encaisses'] += score_B
        stats_groupes[groupe][equipeA]['diff_buts'] += (score_A - score_B)
        
        stats_groupes[groupe][equipeB]['buts_pour'] += score_B
        stats_groupes[groupe][equipeB]['buts_encaisses'] += score_A
        stats_groupes[groupe][equipeB]['diff_buts'] += (score_B - score_A)
        
        if score_A > score_B:
            stats_groupes[groupe][equipeA]['points'] += 3
        elif score_A < score_B:
            stats_groupes[groupe][equipeB]['points'] += 3
        else:
            stats_groupes[groupe][equipeA]['points'] += 1
            stats_groupes[groupe][equipeB]['points'] += 1
print(stats_groupes)


{'A': {'Mexique': {'points': 5, 'buts_pour': 2, 'buts_encaisses': 0, 'diff_buts': 2}, 'Afrique du Sud': {'points': 1, 'buts_pour': 2, 'buts_encaisses': 5, 'diff_buts': -3}, 'Corée du Sud': {'points': 5, 'buts_pour': 2, 'buts_encaisses': 0, 'diff_buts': 2}, 'Tchéquie': {'points': 4, 'buts_pour': 3, 'buts_encaisses': 4, 'diff_buts': -1}}, 'B': {'Canada': {'points': 9, 'buts_pour': 3, 'buts_encaisses': 0, 'diff_buts': 3}, 'Suisse': {'points': 6, 'buts_pour': 4, 'buts_encaisses': 2, 'diff_buts': 2}, 'Qatar': {'points': 3, 'buts_pour': 2, 'buts_encaisses': 4, 'diff_buts': -2}, 'Bosnie-Herzégovine': {'points': 0, 'buts_pour': 0, 'buts_encaisses': 3, 'diff_buts': -3}}, 'C': {'Brésil': {'points': 7, 'buts_pour': 6, 'buts_encaisses': 1, 'diff_buts': 5}, 'Maroc': {'points': 6, 'buts_pour': 3, 'buts_encaisses': 2, 'diff_buts': 1}, 'Haïti': {'points': 0, 'buts_pour': 1, 'buts_encaisses': 8, 'diff_buts': -7}, 'Écosse': {'points': 4, 'buts_pour': 3, 'buts_encaisses': 2, 'diff_buts': 1}}, 'D': {'État

In [97]:
qualifies_directes=[]
troisiemes_en_attente=[]
qualifies_16_eme=[]


In [98]:
for groupe,stat_groupe in stats_groupes.items():
    equipes_tirees=sorted(
        stat_groupe.items(),
        key=lambda x: (x[1]['points'], x[1]['diff_buts'], x[1]['buts_pour']),
        reverse=True
    )
    qualifies_directes.append({"equipe": equipes_tirees[0][0], "groupe":groupe,"rang":1})
    qualifies_directes.append({"equipe": equipes_tirees[1][0], "groupe":groupe,"rang":2})
    troisiemes_en_attente.append({"equipe": equipes_tirees[2][0], "groupe":groupe,"rang":3,"stats":equipes_tirees[2][1]})
print(len(qualifies_directes))
print(len(troisiemes_en_attente))


24
12


In [99]:
meilleur_troisieme_brut=sorted(troisiemes_en_attente,key=lambda x: (x["stats"]['points'], x["stats"]['diff_buts'], x["stats"]['buts_pour']),reverse=True)[:8]
meilleur_troisieme=[]
for equipe in meilleur_troisieme_brut:
    meilleur_troisieme.append({"equipe": equipe["equipe"], "groupe": equipe["groupe"], "rang": 3})


In [100]:
qualifies_16_eme=qualifies_directes+meilleur_troisieme
print(len(qualifies_16_eme))

32


In [ ]:
def generer_16_eme(qualifies_16_eme):
    while True:
        qualif=qualifies_16_eme.copy()
        random.shuffle(qualif)
        match=[]
        tirage_valide=True
        while len(qualif)>0:
            equipeA=qualif.pop(0)
            adversaire_trouve=False
            for i,advr_potentiel in enumerate(qualif):
                if equipeA["groupe"]!=advr_potentiel["groupe"]:
                    equipeB=qualif.pop(i)
                    match.append((equipeA,equipeB))
                    adversaire_trouve=True
                    break
            if not adversaire_trouve:
                tirage_valide=False
                break
        if tirage_valide:
            return(match)
        


In [103]:
matchs_16_eme=generer_16_eme(qualifies_16_eme)
for i,match in enumerate(matchs_16_eme):
    print(f"Match {i+1}: {match[0]['equipe']} vs {match[1]['equipe']}")

Match 1: Canada vs Sénégal
Match 2: Croatie vs Uruguay
Match 3: Argentine vs Brésil
Match 4: Paraguay vs Écosse
Match 5: Mexique vs Égypte
Match 6: Suisse vs Angleterre
Match 7: Iran vs Colombie
Match 8: Tchéquie vs États-Unis
Match 9: Autriche vs Équateur
Match 10: Turquie vs France
Match 11: Espagne vs Corée du Sud
Match 12: Suède vs Maroc
Match 13: Japon vs Congo DR
Match 14: Allemagne vs Belgique
Match 15: Côte d'Ivoire vs Ouzbékistan
Match 16: Pays-Bas vs Arabie Saoudite


In [104]:
def simuler_match_elimination(equipeA,equipeB):
    puissance_attaque_A=df_master[df_master['team']==equipeA]['puissance_attaque'].values[0]
    puissance_attaque_B=df_master[df_master['team']==equipeB]['puissance_attaque'].values[0]
    puissance_defense_A=df_master[df_master['team']==equipeA]['puissance_defense'].values[0]
    puissance_defense_B=df_master[df_master['team']==equipeB]['puissance_defense'].values[0]
    lambda_A=6.0*puissance_attaque_A*(1-puissance_defense_B)
    lambda_B=6.0*puissance_attaque_B*(1-puissance_defense_A)
    score_A=np.random.poisson(lambda_A)
    score_B=np.random.poisson(lambda_B)
    if score_A>score_B:
        return equipeA
    elif score_B>score_A:
        return equipeB
    else:
        return random.choice([equipeA,equipeB])
   

In [106]:
qualifies_8_eme=[]
for match in matchs_16_eme:
    qualifie=simuler_match_elimination(match[0]['equipe'],match[1]['equipe'])
    qualifies_8_eme.append(qualifie)

print(qualifies_8_eme)
print(len(qualifies_8_eme))

['Canada', 'Uruguay', 'Argentine', 'Écosse', 'Mexique', 'Angleterre', 'Colombie', 'États-Unis', 'Autriche', 'France', 'Espagne', 'Maroc', 'Japon', 'Allemagne', 'Ouzbékistan', 'Pays-Bas']
16


In [ ]:
qualifies_quart=[]
